# systemgmmkit quickstart

This notebook is a package-scoped tour of `systemgmmkit`: panel-data setup, robust OLS, fixed/random effects, post-estimation, and panel-aware forecast validation. It uses deterministic simulated data so it can run on Kaggle or Google Colab without external datasets.

What this notebook is: a reproducible cloud demo for users and reviewers.  
What it is not: a full dynamic-GMM parity certificate or paper artifact.

## Install

Kaggle and Colab runtimes are usually clean. If you are running from a local checkout, skip this cell and make sure the source tree is on `PYTHONPATH`.

In [ ]:
# Install the exact public systemgmmkit release used by this notebook.
# Keep --no-deps so Kaggle/Colab scientific packages are not upgraded.
%pip install -q --no-cache-dir --no-deps "universal-output-hub==0.2.4"
%pip uninstall -y -q systemgmmkit
%pip install -q --no-cache-dir --no-deps --force-reinstall "systemgmmkit==1.0.0"

# A notebook rerun can retain modules imported before the reinstall. Remove
# every cached package module so subsequent imports execute the new source.
import importlib
import sys

stale_modules = [
    name
    for name in tuple(sys.modules)
    if name == "systemgmmkit" or name.startswith("systemgmmkit.")
]
for module_name in stale_modules:
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print(f"Cleared {len(stale_modules)} cached systemgmmkit module(s).")


In [ ]:
# Verify that the runtime loaded the warning-free source.
import inspect
from pathlib import Path

import systemgmmkit as sgk
import systemgmmkit.native_gmm as native_gmm

native_path = Path(inspect.getsourcefile(native_gmm) or "")
native_source = native_path.read_text(encoding="utf-8")
lag_start = native_source.index("def _lag_test")
lag_end = native_source.index("candidates:", lag_start)
lag_test_source = native_source[lag_start:lag_end]

print("systemgmmkit version:", getattr(sgk, "__version__", "unknown"))
assert sgk.__version__ == "1.0.0"
assert ".apply(" not in lag_test_source, (
    "Kaggle loaded the deprecated groupby.apply implementation. Rerun the "
    "install cell, then this verification cell."
)
assert "_resid_lag_product" in lag_test_source
print("AR diagnostic implementation: warning-free vectorized version")

## Shared imports and deterministic panel data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import systemgmmkit as sgk
from systemgmmkit.ml import (
    PanelTimeSeriesSplit,
    compare_models,
    cross_validate_panel,
    panel_train_test_split,
)

SEED = 20260730

rng = np.random.default_rng(SEED)

rows = []
for firm in range(1, 31):
    firm_effect = rng.normal(scale=0.35)
    previous_growth = rng.normal(scale=0.2)
    for year in range(2012, 2022):
        investment = rng.normal(loc=1.0 + 0.04 * (year - 2012), scale=0.35)
        leverage = rng.uniform(0.2, 0.8)
        size = rng.normal(loc=firm / 10.0, scale=0.25)
        shock = rng.normal(scale=0.18)
        growth = 0.35 * previous_growth + 0.65 * investment - 0.30 * leverage + 0.12 * size + firm_effect + shock
        rows.append({
            "firm_id": firm,
            "year": year,
            "growth": growth,
            "L1_growth": previous_growth,
            "investment": investment,
            "leverage": leverage,
            "size": size,
        })
        previous_growth = growth

panel = pd.DataFrame(rows)
print(panel.shape)
display(panel.head())

## 1. Robust pooled panel OLS

This is the simplest baseline. It gives users a known starting point before moving to fixed/random effects or GMM specifications.

In [ ]:
pooled_spec = sgk.OLSSpec(
    dependent="growth",
    regressors=["L1_growth", "investment", "leverage", "size"],
    covariance="robust",
    name="pooled_dynamic_panel_ols",
)
pooled = sgk.run_ols(pooled_spec, panel, entity="firm_id", time="year")

post = sgk.quick_postestimation(
    pooled,
    panel,
    y="growth",
    lincoms={"investment_minus_leverage": "investment - leverage"},
    wald_tests={"joint_investment_leverage": "investment = 0, leverage = 0"},
)

display(pooled.params.round(4).to_frame("estimate"))
display(pd.Series(post.metrics).round(6).to_frame("value"))
display(sgk.format_inference_frame(post.linear_combinations, digits=4))
display(sgk.format_inference_frame(post.wald_tests, digits=4))

assert np.isfinite(post.metrics["rmse"])
assert post.linear_combinations is not None
assert post.wald_tests is not None

## 2. Fixed and random effects specifications

The same panel can be estimated with entity-aware specifications. This is the natural bridge from ordinary panel regression to dynamic-panel workflows.

In [ ]:
fe_spec = sgk.FixedEffectsSpec(
    dependent="growth",
    regressors=["investment", "leverage", "size"],
    entity_effects=True,
    covariance="robust",
    name="entity_fixed_effects",
)
re_spec = sgk.RandomEffectsSpec(
    dependent="growth",
    regressors=["investment", "leverage", "size"],
    covariance="robust",
    name="random_effects",
)

fe = sgk.run_fixed_effects(fe_spec, panel, entity="firm_id", time="year")
re = sgk.run_random_effects(re_spec, panel, entity="firm_id", time="year")

# Use a tidy comparison to avoid misleading NaNs from different intercept and
# inferential conventions across pooled, fixed-effects, and random-effects models.
model_comparison = sgk.combine_result_frames(
    [pooled, fe, re],
    model_names=["Pooled OLS", "Fixed effects", "Random effects"],
)
model_comparison = model_comparison.loc[
    model_comparison["term"].isin(["investment", "leverage", "size"]),
    ["model", "term", "coef", "std_err", "p_value"],
].round(4)
display(model_comparison)

assert fe.nobs > 0 and re.nobs > 0
assert set(["investment", "leverage", "size"]).issubset(fe.params.index)
assert not model_comparison.isna().any().any()

## 3. Panel-aware train/test split and cross-validation

The ML layer is useful when the question is predictive performance rather than structural interpretation. The split respects panel time ordering.

In [ ]:
def fit_forecast_model(data: pd.DataFrame):
    return sgk.run_ols(
        sgk.OLSSpec(
            dependent="growth",
            regressors=["L1_growth", "investment", "leverage", "size"],
            covariance="robust",
            name="forecast_ols",
        ),
        data,
        entity="firm_id",
        time="year",
    )

train, test = panel_train_test_split(panel, time="year", test_size=2)
train_result = fit_forecast_model(train)

holdout = compare_models(
    {"dynamic pooled OLS": train_result},
    test,
    y="growth",
    predict_kwargs={"strict": False},
)
cv = cross_validate_panel(
    estimator=fit_forecast_model,
    data=panel,
    y="growth",
    time="year",
    cv=PanelTimeSeriesSplit(n_splits=3, min_train_periods=5, test_periods=1),
    predict_kwargs={"strict": False},
)

display(holdout.round(4))
display(cv.round(4))

assert len(holdout) == 1
assert len(cv) == 3

## 4. Universal Output Hub and package-native reporting

`systemgmmkit` exposes an optional Universal Output Hub adapter for canonical model reporting while retaining its native tidy-table export. This cell adds pooled OLS, fixed-effects, and random-effects results to one OutputHub and writes a Markdown regression table for download from Kaggle/Colab.

In [ ]:
from universal_output_hub import OutputHub

output_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("systemgmmkit_notebook_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

hub = OutputHub("systemgmmkit panel report")
for result, model_name in [
    (pooled, "Pooled OLS"),
    (fe, "Fixed effects"),
    (re, "Random effects"),
]:
    sgk.add_to_outputhub(hub, result, name=model_name, depvar="growth")

report_frame = sgk.combine_result_frames(
    [pooled, fe, re],
    model_names=["Pooled OLS", "Fixed effects", "Random effects"],
)
# Keep the public demo table to common inferential columns. Mixed estimators may
# expose t-statistics, z-statistics, or confidence intervals differently; using
# common fields avoids misleading NaN-heavy presentation.
report_frame = report_frame.loc[
    report_frame["std_err"].notna(),
    ["model", "term", "coef", "std_err", "p_value"],
].round(4)
report_path = output_dir / "systemgmmkit_regression_table.md"
report_path.write_text(report_frame.to_markdown(index=False), encoding="utf-8")

print("OutputHub models:", len(hub.models))
print("Wrote clean regression table:", report_path)
print(report_path.read_text(encoding="utf-8"))
assert len(hub.models) == 3
assert report_path.exists()
assert not report_frame.isna().any().any()

## 5. Difference GMM diagnostic slice

This compact native Difference GMM example demonstrates the dynamic-panel API, instrument construction, and diagnostic fields. It is deliberately small for cloud runtime; production GMM evidence should use the package validation harness and parity artifacts.

In [ ]:
gmm_rows = []
for entity in range(1, 81):
    y_prev = rng.normal(scale=0.3)
    entity_effect = rng.normal(scale=0.2)
    for year in range(1, 12):
        x1 = 0.3 * y_prev + rng.normal(scale=0.5)
        x2 = rng.normal(scale=0.5)
        control = rng.normal(scale=0.5)
        y_value = (
            0.35 * y_prev
            + 0.40 * x1
            - 0.25 * x2
            + 0.15 * control
            + entity_effect
            + rng.normal(scale=0.25)
        )
        gmm_rows.append({
            "entity_id": entity,
            "year": year,
            "y": y_value,
            "x1": x1,
            "x2": x2,
            "control": control,
        })
        y_prev = y_value

gmm_panel = pd.DataFrame(gmm_rows)
gmm_spec = sgk.build_difference_gmm_spec(
    dependent="y",
    regressors=["x1", "x2", "control"],
    endogenous=["x1"],
    predetermined=["x2"],
    exogenous=["control"],
    lag_limits={"y": (2, 3), "x1": (2, 2), "x2": (2, 2)},
    collapse=True,
    transformation="fd",
    steps="twostep",
    time_dummies=False,
    name="notebook_difference_gmm",
)

diff_gmm = sgk.run_difference_gmm(
    gmm_spec,
    gmm_panel,
    entity="entity_id",
    time="year",
    backend="native",
)

gmm_diagnostics = pd.DataFrame([{
    "nobs": diff_gmm.nobs,
    "groups": diff_gmm.n_groups,
    "instruments": diff_gmm.n_instruments,
    "hansen_p": diff_gmm.hansen_p,
    "sargan_p": diff_gmm.sargan_p,
    "ar1_p": diff_gmm.ar1_p,
    "ar2_p": diff_gmm.ar2_p,
}])

display(diff_gmm.params.round(4).to_frame("estimate"))
display(gmm_diagnostics.round(4))
instrument_health = diff_gmm.check_instrument_health()
print(instrument_health.to_markdown())
print(diff_gmm.to_markdown()[:1600])

assert diff_gmm.nobs > 0
assert diff_gmm.n_instruments <= diff_gmm.n_groups
assert instrument_health.status == "acceptable"
assert diff_gmm.ar2_p is None or diff_gmm.ar2_p > 0.05

gmm_hub_model = sgk.add_to_outputhub(
    hub,
    diff_gmm,
    name="Difference GMM",
    depvar="y",
    include_diagnostics=True,
)
print("OutputHub models after GMM:", len(hub.models))
print("OutputHub diagnostic tables:", len(hub.tables))
display(hub.tables[0].data.round(4))
assert gmm_hub_model.metadata["estimator"] == "difference_gmm"
assert len(hub.models) == 4
assert len(hub.tables) == 1

## 6. Interpretation checklist

For a public notebook, keep the claims clear:

- pooled, fixed-effects, random-effects, and dynamic-GMM estimates answer different questions;
- panel-aware validation is about forecasting discipline, not causal identification;
- the reporting/export cell writes a reusable Markdown table for Kaggle/Colab outputs;
- dynamic GMM validation requires AR tests, Hansen/Sargan evidence, instrument count checks, and parity artifacts, which belong in the package validation and paper workflow.